<a href="https://colab.research.google.com/github/sakaleshhubli/sem7-ML2-/blob/main/4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 🧠 Concept Learning Explorer: Find-S & Candidate-Elimination {display-mode: "form"}

import json
import pandas as pd
from IPython.display import HTML
from google.colab import output

# Default classic EnjoySport dataset
default_data = [
    {"Sky": "Sunny", "Temp": "Warm", "Humid": "Normal", "Wind": "Strong", "Water": "Warm", "Forecast": "Same", "EnjoySport": "Yes"},
    {"Sky": "Sunny", "Temp": "Warm", "Humid": "High", "Wind": "Strong", "Water": "Warm", "Forecast": "Same", "EnjoySport": "Yes"},
    {"Sky": "Rainy", "Temp": "Cold", "Humid": "High", "Wind": "Strong", "Water": "Warm", "Forecast": "Change", "EnjoySport": "No"},
    {"Sky": "Sunny", "Temp": "Warm", "Humid": "High", "Wind": "Strong", "Water": "Cool", "Forecast": "Change", "EnjoySport": "Yes"}
]

def run_algorithms(examples):
    if not examples:
        return {"steps_ce": [], "steps_find_s": [], "final_s_ce": [], "final_g_ce": [], "final_find_s": []}

    attributes = [k for k in examples[0].keys() if k != "EnjoySport"]
    num_attrs = len(attributes)

    # ==========================================
    # 1. RUN FIND-S (Only uses positive examples)
    # ==========================================
    find_s_hyp = ["0"] * num_attrs
    steps_find_s = []
    pos_count = 0

    for idx, ex in enumerate(examples):
        instance = [ex[attr] for attr in attributes]
        label = ex["EnjoySport"]

        prev_hyp = list(find_s_hyp)
        action = "Ignored (Negative Example)"

        if label == "Yes":
            pos_count += 1
            if find_s_hyp == ["0"] * num_attrs:
                find_s_hyp = list(instance)
                action = "Initialized with 1st positive instance"
            else:
                action = "Generalized attributes: "
                diff_fields = []
                for i in range(num_attrs):
                    if find_s_hyp[i] != instance[i]:
                        diff_fields.append(attributes[i])
                        find_s_hyp[i] = "?"
                if diff_fields:
                    action += ", ".join(diff_fields) + " -> '?'"
                else:
                    action = "No changes (Instance already satisfies hypothesis)"

        steps_find_s.append({
            "step": idx + 1,
            "instance": ex,
            "hyp_before": prev_hyp,
            "hyp_after": list(find_s_hyp),
            "action": action
        })

    # ==========================================
    # 2. RUN CANDIDATE-ELIMINATION
    # ==========================================
    S = ["0"] * num_attrs
    G = [["?"] * num_attrs]
    steps_ce = []

    def satisfies(instance, hypothesis):
        for i, val in enumerate(hypothesis):
            if val != "?" and val != instance[i]:
                return False
        return True

    for idx, ex in enumerate(examples):
        instance = [ex[attr] for attr in attributes]
        label = ex["EnjoySport"]

        prev_s = list(S)
        prev_g = [list(g) for g in G]

        if label == "Yes": # Positive example
            if S == ["0"] * num_attrs:
                S = list(instance)
            else:
                for i in range(num_attrs):
                    if S[i] != instance[i]:
                        S[i] = "?"
            G = [g for g in G if satisfies(instance, g)]
        else: # Negative example
            new_G = []
            for g in G:
                if satisfies(instance, g):
                    for i in range(num_attrs):
                        if g[i] == "?" and S[i] != "0" and S[i] != "?" and S[i] != instance[i]:
                            g_special = list(g)
                            g_special[i] = S[i]
                            if g_special not in new_G:
                                new_G.append(g_special)
                else:
                    if g not in new_G:
                        new_G.append(g)

            G = []
            for g in new_G:
                consistent_with_S = True
                for i in range(num_attrs):
                    if g[i] != "?" and g[i] != S[i]:
                        consistent_with_S = False
                        break
                if consistent_with_S:
                    G.append(g)

            if not G:
                G = [["?"] * num_attrs]

        steps_ce.append({
            "step": idx + 1,
            "instance": ex,
            "S_before": prev_s,
            "G_before": prev_g,
            "S_after": list(S),
            "G_after": [list(g) for g in G]
        })

    return {
        "attributes": attributes,
        "steps_find_s": steps_find_s,
        "steps_ce": steps_ce,
        "final_find_s": find_s_hyp,
        "final_s_ce": S,
        "final_g_ce": G
    }

def process_data_callback(json_str):
    try:
        examples = json.loads(json_str)
        results = run_algorithms(examples)
        return json.dumps(results)
    except Exception as e:
        return json.dumps({"error": str(e)})

output.register_callback('process_data', process_data_callback)

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Find-S vs Candidate-Elimination Dashboard</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f4f6f8;
            color: #1e293b;
            margin: 0;
            padding: 20px;
        }
        .dashboard-container {
            max-width: 1400px;
            margin: 0 auto;
            display: flex;
            flex-direction: column;
            gap: 20px;
        }
        header {
            background: linear-gradient(135deg, #0f172a, #1e293b);
            color: white;
            padding: 24px;
            border-radius: 12px;
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
        }
        header h1 {
            margin: 0 0 8px 0;
            font-size: 24px;
            font-weight: 700;
        }
        header p {
            margin: 0;
            font-size: 14px;
            opacity: 0.9;
        }
        .main-layout {
            display: grid;
            grid-template-columns: 350px 1fr;
            gap: 20px;
        }
        @media (max-width: 1000px) {
            .main-layout {
                grid-template-columns: 1fr;
            }
        }
        .card {
            background: white;
            border-radius: 12px;
            padding: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            display: flex;
            flex-direction: column;
            gap: 15px;
        }
        .card-title {
            font-size: 16px;
            font-weight: 600;
            color: #0f172a;
            border-bottom: 1px solid #e2e8f0;
            padding-bottom: 10px;
            margin: 0;
        }
        .btn {
            background-color: #4f46e5;
            color: white;
            border: none;
            padding: 10px 16px;
            border-radius: 6px;
            font-weight: 500;
            cursor: pointer;
            transition: background 0.2s;
            text-align: center;
        }
        .btn:hover {
            background-color: #4338ca;
        }
        .btn-secondary {
            background-color: #e2e8f0;
            color: #334155;
        }
        .btn-secondary:hover {
            background-color: #cbd5e1;
        }
        .table-wrapper {
            overflow-x: auto;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            text-align: left;
            font-size: 13px;
        }
        th, td {
            padding: 10px;
            border-bottom: 1px solid #e2e8f0;
        }
        th {
            background-color: #f8fafc;
            font-weight: 600;
            color: #64748b;
        }
        .badge {
            padding: 4px 8px;
            border-radius: 4px;
            font-size: 11px;
            font-weight: 600;
        }
        .badge-yes {
            background-color: #dcfce7;
            color: #166534;
        }
        .badge-no {
            background-color: #fee2e2;
            color: #991b1b;
        }
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 15px;
        }
        .kpi-card {
            background: #f8fafc;
            border: 1px solid #e2e8f0;
            border-radius: 8px;
            padding: 15px;
            text-align: center;
        }
        .kpi-value {
            font-family: monospace;
            font-size: 13px;
            font-weight: 700;
            color: #4f46e5;
            word-break: break-all;
            margin-top: 8px;
        }
        .results-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }
        @media (max-width: 768px) {
            .results-grid {
                grid-template-columns: 1fr;
            }
        }
        .step-container {
            display: flex;
            flex-direction: column;
            gap: 12px;
        }
        .step-block {
            border-left: 4px solid #3b82f6;
            background-color: #f8fafc;
            padding: 12px;
            border-radius: 0 8px 8px 0;
        }
        .step-header {
            font-weight: 600;
            margin-bottom: 6px;
            display: flex;
            justify-content: space-between;
            font-size: 13px;
        }
        .step-hypo {
            font-family: monospace;
            background: #ffffff;
            border: 1px solid #e2e8f0;
            padding: 4px 8px;
            border-radius: 4px;
            margin-top: 4px;
            font-size: 11px;
            display: inline-block;
        }
        .form-group {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(80px, 1fr));
            gap: 8px;
            margin-bottom: 8px;
        }
        label {
            font-size: 11px;
            font-weight: 500;
            color: #64748b;
            display: block;
            margin-bottom: 4px;
        }
        input, select {
            width: 100%;
            padding: 6px;
            border-radius: 4px;
            border: 1px solid #cbd5e1;
            font-size: 11px;
            box-sizing: border-box;
        }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <header>
            <h1>🧠 Find-S vs. Candidate-Elimination Solver</h1>
            <p>Interactively contrast the Specific-to-General Find-S heuristic against the General-to-Specific Candidate-Elimination Version Space algorithm.</p>
        </header>

        <div class="main-layout">
            <!-- Controls Column -->
            <div style="display: flex; flex-direction: column; gap: 20px;">
                <div class="card">
                    <h3 class="card-title">➕ Add Instance</h3>
                    <form id="add-instance-form">
                        <div class="form-group">
                            <div>
                                <label>Sky</label>
                                <select id="attr-sky">
                                    <option value="Sunny">Sunny</option>
                                    <option value="Cloudy">Cloudy</option>
                                    <option value="Rainy">Rainy</option>
                                </select>
                            </div>
                            <div>
                                <label>Temp</label>
                                <select id="attr-temp">
                                    <option value="Warm">Warm</option>
                                    <option value="Cold">Cold</option>
                                </select>
                            </div>
                        </div>
                        <div class="form-group">
                            <div>
                                <label>Humidity</label>
                                <select id="attr-humid">
                                    <option value="Normal">Normal</option>
                                    <option value="High">High</option>
                                </select>
                            </div>
                            <div>
                                <label>Wind</label>
                                <select id="attr-wind">
                                    <option value="Strong">Strong</option>
                                    <option value="Weak">Weak</option>
                                </select>
                            </div>
                        </div>
                        <div class="form-group">
                            <div>
                                <label>Water</label>
                                <select id="attr-water">
                                    <option value="Warm">Warm</option>
                                    <option value="Cool">Cool</option>
                                </select>
                            </div>
                            <div>
                                <label>Forecast</label>
                                <select id="attr-forecast">
                                    <option value="Same">Same</option>
                                    <option value="Change">Change</option>
                                </select>
                            </div>
                        </div>
                        <div class="form-group">
                            <div style="grid-column: span 2;">
                                <label>EnjoySport (Target)</label>
                                <select id="attr-target" style="font-weight: bold;">
                                    <option value="Yes">Yes (+ve)</option>
                                    <option value="No">No (-ve)</option>
                                </select>
                            </div>
                        </div>
                        <button type="button" class="btn" onclick="addInstance()" style="width: 100%; margin-top: 5px;">Add Instance</button>
                    </form>
                </div>

                <div class="card">
                    <h3 class="card-title">📋 Current Dataset Table</h3>
                    <div class="table-wrapper">
                        <table id="dataset-table">
                            <thead>
                                <tr>
                                    <th>Sky</th>
                                    <th>Temp</th>
                                    <th>Humid</th>
                                    <th>Wind</th>
                                    <th>Water</th>
                                    <th>Fore</th>
                                    <th>Target</th>
                                    <th>Action</th>
                                </tr>
                            </thead>
                            <tbody></tbody>
                        </table>
                    </div>
                    <button class="btn btn-secondary" onclick="resetToDefault()">Reset to Default</button>
                </div>
            </div>

            <!-- Results & Visualization Column -->
            <div style="display: flex; flex-direction: column; gap: 20px;">
                <!-- Overview Dashboard KPI Cards -->
                <div class="card">
                    <h3 class="card-title">🎯 Comparative Hypothesis Boundaries</h3>
                    <div class="kpi-row">
                        <div class="kpi-card" style="border-top: 4px solid #6366f1;">
                            <label style="font-size: 13px; font-weight: 600; color: #4338ca;">Find-S Hypothesis</label>
                            <div id="final-find-s" class="kpi-value">0</div>
                        </div>
                        <div class="kpi-card" style="border-top: 4px solid #10b981;">
                            <label style="font-size: 13px; font-weight: 600; color: #047857;">Candidate-Elimination S</label>
                            <div id="final-s" class="kpi-value">0</div>
                        </div>
                        <div class="kpi-card" style="border-top: 4px solid #ef4444;">
                            <label style="font-size: 13px; font-weight: 600; color: #b91c1c;">Candidate-Elimination G</label>
                            <div id="final-g" class="kpi-value">?</div>
                        </div>
                    </div>
                </div>

                <!-- Step-by-Step Executions side-by-side -->
                <div class="results-grid">
                    <!-- Find-S Side -->
                    <div class="card">
                        <h3 class="card-title">🔍 Find-S Progression (Positives Only)</h3>
                        <div class="step-container" id="find-s-trace"></div>
                    </div>

                    <!-- Candidate Elimination Side -->
                    <div class="card">
                        <h3 class="card-title">⚖️ Candidate-Elimination Trace</h3>
                        <div class="step-container" id="ce-trace"></div>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        let currentDataset = JSON.parse('DEFAULT_DATA_PLACEHOLDER');

        function updateUI() {
            const tbody = document.querySelector('#dataset-table tbody');
            tbody.innerHTML = '';
            currentDataset.forEach((row, index) => {
                const tr = document.createElement('tr');
                tr.innerHTML = `
                    <td>${row.Sky}</td>
                    <td>${row.Temp}</td>
                    <td>${row.Humid}</td>
                    <td>${row.Wind}</td>
                    <td>${row.Water}</td>
                    <td>${row.Forecast}</td>
                    <td><span class="badge badge-${row.EnjoySport.toLowerCase()}">${row.EnjoySport}</span></td>
                    <td><button class="btn btn-secondary" style="padding: 2px 6px; font-size: 10px;" onclick="deleteRow(${index})">❌</button></td>
                `;
                tbody.appendChild(tr);
            });

            google.colab.kernel.invokeFunction('process_data', [JSON.stringify(currentDataset)], {})
                .then(response => {
                    const data = JSON.parse(response.data['text/plain']);
                    if (data.error) {
                        console.error(data.error);
                        return;
                    }
                    renderResults(data);
                });
        }

        function renderResults(data) {
            // Update KPI blocks
            document.getElementById('final-find-s').innerText = `h = <${data.final_find_s.join(', ')}>`;
            document.getElementById('final-s').innerText = `S = <${data.final_s_ce.join(', ')}>`;
            document.getElementById('final-g').innerText = `G = { ${data.final_g_ce.map(g => `<${g.join(', ')}>`).join(' , ')} }`;

            // Render Find-S progression
            const findSContainer = document.getElementById('find-s-trace');
            findSContainer.innerHTML = '';
            data.steps_find_s.forEach(step => {
                const isPos = step.instance.EnjoySport === 'Yes';
                const stepBlock = document.createElement('div');
                stepBlock.className = 'step-block';
                stepBlock.style.borderLeftColor = isPos ? '#10b981' : '#cbd5e1';

                stepBlock.innerHTML = `
                    <div class="step-header">
                        <span>Step ${step.step}: <${Object.values(step.instance).slice(0, -1).join(', ')}></span>
                        <span class="badge badge-${step.instance.EnjoySport.toLowerCase()}">${step.instance.EnjoySport}</span>
                    </div>
                    <div style="font-size: 11px; margin-top: 4px;">
                        <div><strong>Hypothesis:</strong> <span class="step-hypo">&lt;${step.hyp_after.join(', ')}&gt;</span></div>
                        <div style="color: #64748b; margin-top: 4px; font-style: italic;">${step.action}</div>
                    </div>
                `;
                findSContainer.appendChild(stepBlock);
            });

            // Render Candidate-Elimination progression
            const ceContainer = document.getElementById('ce-trace');
            ceContainer.innerHTML = '';
            data.steps_ce.forEach(step => {
                const stepBlock = document.createElement('div');
                stepBlock.className = 'step-block';
                stepBlock.style.borderLeftColor = '#3b82f6';

                stepBlock.innerHTML = `
                    <div class="step-header">
                        <span>Step ${step.step}: <${Object.values(step.instance).slice(0, -1).join(', ')}></span>
                        <span class="badge badge-${step.instance.EnjoySport.toLowerCase()}">${step.instance.EnjoySport}</span>
                    </div>
                    <div style="font-size: 11px; margin-top: 4px;">
                        <div><strong>S:</strong> <span class="step-hypo">&lt;${step.S_after.join(', ')}&gt;</span></div>
                        <div style="margin-top: 4px;"><strong>G:</strong> <span class="step-hypo">{ ${step.G_after.map(g => `&lt;${g.join(', ')}&gt;`).join(', ')} }</span></div>
                    </div>
                `;
                ceContainer.appendChild(stepBlock);
            });
        }

        function addInstance() {
            const newRow = {
                Sky: document.getElementById('attr-sky').value,
                Temp: document.getElementById('attr-temp').value,
                Humid: document.getElementById('attr-humid').value,
                Wind: document.getElementById('attr-wind').value,
                Water: document.getElementById('attr-water').value,
                Forecast: document.getElementById('attr-forecast').value,
                EnjoySport: document.getElementById('attr-target').value
            };
            currentDataset.push(newRow);
            updateUI();
        }

        function deleteRow(index) {
            currentDataset.splice(index, 1);
            updateUI();
        }

        function resetToDefault() {
            currentDataset = JSON.parse('DEFAULT_DATA_PLACEHOLDER');
            updateUI();
        }

        // Init
        updateUI();
    </script>
</body>
</html>
""".replace('DEFAULT_DATA_PLACEHOLDER', json.dumps(default_data))

display(HTML(html_content))


Sky,Temp,Humid,Wind,Water,Fore,Target,Action
